In [1]:
import sys; print(sys.executable)
import tensorflow as tf
from dense_pcn_layer import DensePCNLayer
from conv_pcn_layer import Conv2DPCNLayer
from transformer_pcn_layer import TransformerPCNLayer, PositionalEncodingLayer
%load_ext autoreload
%autoreload 2

c:\Users\user\miniconda3\envs\tf_py3.10\python.exe


In [4]:
tf.__version__

'2.10.0'

In [5]:
tf.config.set_visible_devices([], 'GPU')

In [ ]:
from urllib.request import urlretrieve
urlretrieve("http://images.cocodataset.org/zips/train2017.zip", "train2017.zip")
urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", "coco.zip")

KeyboardInterrupt: 

In [ ]:
import zipfile
zipfile.ZipFile('train2017.zip', 'r').extractall('train2017')
zipfile.ZipFile('coco.zip', 'r').extractall('coco')

In [6]:
import json
train_captions = json.load(open("coco/annotations/captions_train2017.json", 'r'))

In [7]:
import cv2
image_data = []
image_data_id = []
for image in train_captions['images'][:100]:
    img = cv2.imread(f"train2017/train2017/{image['file_name']}")
    img = cv2.resize(img, (572, 572))
    image_data.append(img)
    image_data_id.append(image["id"])

In [8]:
import numpy as np
image_data = np.array(image_data)
image_data_id = np.array(image_data_id)

In [9]:
caption_data = []
caption_data_image_id =[]
for annotation in train_captions['annotations']:
    if (image_data_id == annotation['image_id']).sum() == 1:
        caption_data.append(annotation['caption'])
        caption_data_image_id.append(annotation['image_id'])
caption_data = np.array(caption_data)
caption_data_image_id = np.array(caption_data_image_id)

In [10]:
chars = set()
for caption in caption_data:
    for char in caption:
        chars.add(char)

In [11]:
chars.add('\0')
chars_arr = np.array(list(chars))

In [12]:
num_tokens = np.array([len(caption) for caption in caption_data]).max()

In [13]:
print(num_tokens)
num_tokens=192

150


In [14]:
tokenized = []
for caption in caption_data:
    seq = []
    for i in range(num_tokens):
        try:
            seq.append(chars_arr == caption[i])
        except IndexError:
            seq.append(chars_arr == '\0')
    tokenized.append(seq)

In [15]:
print(len(chars_arr))

52


In [16]:
tokenized_captions = np.array(tokenized)

In [17]:
caption_matching_images = []
for id in caption_data_image_id:
    caption_matching_images.append(image_data[image_data_id==id])
caption_matching_images = np.array(caption_matching_images)

In [18]:
np.where(chars_arr == '\0')

(array([11], dtype=int64),)

In [19]:
print(tokenized_captions.shape)

(501, 192, 52)


In [20]:
mask =  tf.where(tokenized_captions[:, :, np.where(chars_arr == '\0')], tf.cast(-1e9, tf.float32), tf.cast(0.0, tf.float32)) 

In [21]:
mask =tf.reshape(mask, (mask.shape[0], mask.shape[1]))

In [22]:
mask

<tf.Tensor: shape=(501, 192), dtype=float32, numpy=
array([[ 0.e+00,  0.e+00,  0.e+00, ..., -1.e+09, -1.e+09, -1.e+09],
       [ 0.e+00,  0.e+00,  0.e+00, ..., -1.e+09, -1.e+09, -1.e+09],
       [ 0.e+00,  0.e+00,  0.e+00, ..., -1.e+09, -1.e+09, -1.e+09],
       ...,
       [ 0.e+00,  0.e+00,  0.e+00, ..., -1.e+09, -1.e+09, -1.e+09],
       [ 0.e+00,  0.e+00,  0.e+00, ..., -1.e+09, -1.e+09, -1.e+09],
       [ 0.e+00,  0.e+00,  0.e+00, ..., -1.e+09, -1.e+09, -1.e+09]],
      dtype=float32)>

In [36]:
import gc
gc.collect()

0

In [27]:
A = DensePCNLayer(128, 0.00001)
C = DensePCNLayer(200, 0.00001)
A.state = tf.Variable(tf.random.normal((1, 192, 128)))
B = TransformerPCNLayer(3, 128, 8, 0.00001, A, [C], mask[0][None])
A.next_layers=[B]
C.prev_layer = B
b_out = B(A.state)
c_out = C(b_out)

In [23]:
for i in range(10):
    for layer in B.get_layers():
        layer.update_state()
        layer.update_wts()
        layer.update_b()
    print(tf.reduce_sum((B.predict_next() - B(A.predict_next()))**2)
        +tf.reduce_sum((C.predict_next() - C(B.predict_next()))**2)
        +tf.reduce_sum((A.predict_next()-B.predict_prev())**2)
        +tf.reduce_sum((B.predict_next()-C.predict_prev())**2))

tf.Tensor(225544.53, shape=(), dtype=float32)
tf.Tensor(222998.22, shape=(), dtype=float32)
tf.Tensor(220503.75, shape=(), dtype=float32)
tf.Tensor(218059.17, shape=(), dtype=float32)
tf.Tensor(215662.56, shape=(), dtype=float32)
tf.Tensor(213312.34, shape=(), dtype=float32)
tf.Tensor(211006.98, shape=(), dtype=float32)
tf.Tensor(208745.19, shape=(), dtype=float32)
tf.Tensor(206525.73, shape=(), dtype=float32)
tf.Tensor(204347.45, shape=(), dtype=float32)


In [30]:
del A, B, C

In [21]:
caption_matching_images.shape

(501, 1, 572, 572, 3)

In [28]:
tokenized_captions_shuffled.shape

(100, 192, 52)

In [25]:
from encoder_encoder_pcn import EncoderEncoderPCN
eepcn = EncoderEncoderPCN(1e-4)
# eepcn.pass_through(tf.cast(tf.convert_to_tensor(caption_matching_images[0]), tf.float32), tf.cast(tf.convert_to_tensor(tokenized_captions[[0]]), tf.float32), mask[0][None])

In [26]:
inds = np.arange(caption_matching_images.shape[0])
np.random.shuffle(inds)

In [27]:
caption_matching_images_shuffled = caption_matching_images[inds[:100]]
tokenized_captions_shuffled = tokenized_captions[inds[:100]]

In [28]:
mask_shuffled = tf.gather(mask, inds[:100], axis=0)

In [31]:
caption_matching_images_shuffled[0].shape, tokenized_captions_shuffled[0].shape

(TensorShape([1, 572, 572, 3]), TensorShape([1, 192, 52]))

In [32]:
caption_matching_images_shuffled[0].shape, tokenized_captions_shuffled[0].shape, mask_shuffled[0].shape

(TensorShape([1, 572, 572, 3]),
 TensorShape([1, 192, 52]),
 TensorShape([1, 192]))

In [ ]:
eepcn.train_step(1, tf.convert_to_tensor(caption_matching_images_shuffled[0], dtype=tf.float32), tf.convert_to_tensor(tokenized_captions_shuffled[[0]], dtype=tf.float32), mask_shuffled[0][None])

(1, 102400)
(1, 161817)
(1, 345871)
(1, 702332)
(1, 1429912)
(1, 102400)
(1, 161817)
(1, 345871)
(1, 702332)
(1, 1429912)


ResourceExhaustedError: {{function_node __wrapped__Transpose_device_/job:localhost/replica:0/task:0/device:CPU:0}} OOM when allocating tensor with shape[100,20647936] and type float on /job:localhost/replica:0/task:0/device:CPU:0 by allocator mklcpu [Op:Transpose]

In [26]:
for i in range(100):
    print(f"{i+1}/100")
    eepcn.train_step(10, caption_matching_images_shuffled[i], tokenized_captions_shuffled[i], mask_shuffled[i])
    gc.collect()

1/100
(1, 102400)
(1, 161817)
(1, 345871)
(1, 702332)
(1, 1429912)
(1, 102400)
(1, 161817)
(1, 345871)
(1, 702332)
(1, 1429912)


: 

In [25]:
txt_out = eepcn.test_step(1, tf.cast(caption_matching_images[0], tf.float32), tf.zeros_like(tf.cast(tf.convert_to_tensor(tokenized_captions[[0]]), tf.float32)), predict='txt')

(1, 102400)
(1, 161817)
(1, 345871)
(1, 702332)
(1, 1429912)
(1, 102400)
(1, 161817)
(1, 345871)
(1, 702332)
(1, 1429912)


In [26]:
print(''.join(chars_arr[tokenized_captions[0].argmax(axis=-1)]))

Rows of motor bikes and helmets in a city


In [ ]:
print(''.join(chars_arr[txt_out[0].numpy().argmax(axis=-1)]))

oooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooo


hi


In [ ]:
print("what")

what
